In [22]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import openpyxl
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")

# ============================================================
# DB CONFIG
# ============================================================
DB_CONFIG = {
    "drivername": "postgresql+psycopg2",
    "username": "postgres",
    "password": "Volkswagengolf2025",
    "host": "localhost",
    "port": 5432,
    "database": "NML_db",
}

DB_URL = (
    f"{DB_CONFIG['drivername']}://{DB_CONFIG['username']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

# ============================================================
# BASIC HELPERS
# ============================================================
def safe_numeric(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    s = str(value).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return None
    s = s.replace(",", "")
    try:
        return float(s)
    except Exception:
        return None


def safe_date_convert(value):
    """
    Robust:
    - handles real datetime-like values from Excel
    - handles numeric excel serial dates
    - handles strings
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None

    if isinstance(value, pd.Timestamp):
        try:
            return value.date()
        except Exception:
            pass

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        try:
            dt = pd.to_datetime(value, unit="d", origin="1899-12-30", errors="coerce")
            return dt.date() if pd.notna(dt) else None
        except Exception:
            pass

    s = str(value).strip()
    if s in ("", "nan", "NaN", "None"):
        return None
    try:
        dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
        return dt.date() if pd.notna(dt) else None
    except Exception:
        return None


def clean_value(val):
    if val is None:
        return None
    s = str(val).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return None
    return s


# ============================================================
# ADMIN SHEET LABEL HELPERS
# ============================================================
def _cell(df, r, c):
    try:
        v = df.iat[r, c]
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        s = str(v).strip()
        return None if s in ("", "nan", "NaN", "None") else s
    except Exception:
        return None


def _find_label_positions(df, label_regex):
    hits = []
    rx = re.compile(label_regex, re.IGNORECASE)
    for r in range(df.shape[0]):
        row = df.iloc[r, :]
        for c in range(df.shape[1]):
            v = row.iat[c]
            if isinstance(v, str) and rx.search(v.strip()):
                hits.append((r, c))
    return hits


def _value_right(df, r, c, max_look=6):
    for k in range(1, max_look + 1):
        val = _cell(df, r, c + k)
        if val:
            return val
    return None


def _value_below(df, r, c, max_look=6):
    for k in range(1, max_look + 1):
        val = _cell(df, r + k, c)
        if val:
            return val
    return None


def get_labeled_value(df, label_regex, right_look=6, below_look=6):
    hits = _find_label_positions(df, rf"^{label_regex}\s*:?\s*$")
    for (r, c) in hits:
        val = _value_right(df, r, c, right_look)
        if not val:
            val = _value_below(df, r, c, below_look)
        if val:
            return val
    return None


def get_block_below(df, label_regex, below_rows=6):
    hits = _find_label_positions(df, rf"^{label_regex}\s*:?\s*$")
    for (r, c) in hits:
        parts = []
        for k in range(1, below_rows + 1):
            v = _cell(df, r + k, c)
            if v:
                parts.append(v)
        if parts:
            return " ".join(parts)
    return None


def extract_admin_df(file_path):
    xl = pd.ExcelFile(file_path)
    sheet_name = "Admin" if "Admin" in xl.sheet_names else xl.sheet_names[0]
    return pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=object)


def extract_gauge_id(file_path):
    try:
        df = extract_admin_df(file_path)
        gauge = get_labeled_value(df, r"(Gauge\s*ID|Instrument\s*Details|Instrument\s*ID)")
        return clean_value(gauge)
    except Exception:
        return None


def extract_metadata(file_path):
    """
    Universal metadata only (no mass fields).
    """
    try:
        df = extract_admin_df(file_path)
        md = {"file_name": os.path.basename(file_path)}

        md["certificate_number"] = clean_value(
            get_labeled_value(df, r"Certificate\s*Number")
            or get_labeled_value(df, r"Cert(\.|ificate)?\s*No")
        )

        md["calibration_date"] = safe_date_convert(
            get_labeled_value(df, r"Calibration\s*Date")
            or get_labeled_value(df, r"Date\s*of\s*Calibration")
            or get_labeled_value(df, r"Calibration\s*Dates?")
            or get_labeled_value(df, r"Mean\s*Date")
            or get_labeled_value(df, r"Start\s*Date")
            or get_labeled_value(df, r"End\s*Date")
            or get_labeled_value(df, r"Date")
        )

        md["manufacturer"] = clean_value(get_labeled_value(df, r"Manufacturer"))
        md["instrument_type"] = clean_value(get_labeled_value(df, r"Instrument\s*Type"))
        md["serial_number"] = clean_value(get_labeled_value(df, r"Serial\s*Number"))
        md["previous_certificate_number"] = clean_value(get_labeled_value(df, r"Previous\s*Certificate\s*Number"))
        md["date_received"] = safe_date_convert(get_labeled_value(df, r"Date\s*Received"))

        md["calibration_procedure_number"] = clean_value(
            get_labeled_value(df, r"(Calibration\s*Procedure\s*Numbers?|NML\s+Procedure\s*Number|(NML\s+)?Procedure\s*Number)")
        )

        md["method_used"] = clean_value(
            get_labeled_value(df, r"De.?cription of Method Used")
            or get_labeled_value(df, r"Method Used")
            or get_labeled_value(df, r"Method")
        )

        md["comments"] = clean_value(
            get_labeled_value(df, r"Comments") or get_block_below(df, r"Comments", below_rows=8)
        )

        return md
    except Exception as e:
        print(f"⚠️ Error extracting metadata from {os.path.basename(file_path)}: {e}")
        return {"file_name": os.path.basename(file_path)}


# ============================================================
# UNCERTAINTY (DIMENSIONAL)
# ============================================================
def find_uncertainty_from_workbook(file_path: str):
    """
    Supports:
      - Constant: 'The uncertainty of measurement is ± 0.0007 Inches'
      - Formula:  'The uncertainty of measurement is ± (2.3+L/300) µm'
    Returns:
      - {"type":"constant", "value": float, "unit": unit_or_None}
      - {"type":"formula_um", "a": float, "b": float}
      - None
    """
    try:
        wb = openpyxl.load_workbook(file_path, data_only=True, keep_vba=True)
        for sh in wb.sheetnames[:15]:
            ws = wb[sh]
            for r in range(1, 300):
                v = ws.cell(r, 1).value
                if not isinstance(v, str):
                    continue
                low = v.lower()
                if "uncertainty of measurement" not in low:
                    continue

                txt = v.strip()

                m = re.search(
                    r"uncertainty of measurement\s+is\s*±?\s*([0-9]*\.?[0-9]+)\s*(mm|inch|inches|µm|um)?",
                    txt,
                    re.I,
                )
                if m:
                    unit = m.group(2).lower() if m.group(2) else None
                    return {"type": "constant", "value": float(m.group(1)), "unit": unit}

                m = re.search(
                    r"±?\s*\(?\s*([0-9]*\.?[0-9]+)\s*\+\s*L\s*/\s*([0-9]*\.?[0-9]+)\s*\)?\s*(µm|um)",
                    txt,
                    re.I,
                )
                if m:
                    return {"type": "formula_um", "a": float(m.group(1)), "b": float(m.group(2))}

        return None
    except Exception as e:
        print(f"   ⚠️ Error scanning for uncertainty: {e}")
        return None


def compute_measurement_uncertainty(unc_spec, nominal_value, target_unit):
    if unc_spec is None or nominal_value is None:
        return None
    try:
        L = float(nominal_value)
    except Exception:
        return None

    if unc_spec["type"] == "formula_um":
        a = float(unc_spec["a"])
        b = float(unc_spec["b"])
        u_um = a + (L / b)
        if target_unit == "mm":
            return u_um * 1e-3
        if target_unit == "inch":
            return (u_um * 1e-6) / 0.0254
        return u_um

    if unc_spec["type"] == "constant":
        val = float(unc_spec["value"])
        u = unc_spec.get("unit")
        if u in ("µm", "um"):
            if target_unit == "mm":
                return val * 1e-3
            if target_unit == "inch":
                return (val * 1e-6) / 0.0254
            return val
        if u is None:
            return val
        if u == "inches":
            u = "inch"
        if u == target_unit:
            return val
        if u == "inch" and target_unit == "mm":
            return val * 25.4
        if u == "mm" and target_unit == "inch":
            return val / 25.4
        return val

    return None


# ============================================================
# DIMENSIONAL RESULTS (MULTI-TEMPLATE SAFE)
# ============================================================
def detect_unit(df_raw, header_row: int):
    scan_rows = [header_row - 1, header_row, header_row + 1, header_row + 2, header_row + 3]
    for r in scan_rows:
        if r < 0 or r >= df_raw.shape[0]:
            continue
        for v in df_raw.iloc[r]:
            if isinstance(v, str):
                low = v.lower()
                if "mm" in low:
                    return "mm"
                if "inch" in low or "inches" in low:
                    return "inch"
    return None


def extract_feature_name(df_raw, header_row: int):
    candidates = []
    start = max(0, header_row - 25)
    for r in range(start, header_row):
        for v in df_raw.iloc[r]:
            if not isinstance(v, str):
                continue
            s = v.strip()
            if not s:
                continue
            low = s.lower()
            if low.startswith("file:"):
                continue
            if "uncertainty" in low:
                continue
            if any(
                k in low
                for k in [
                    "nominal value",
                    "measured value",
                    "deviation",
                    "tolerance",
                    "results table",
                    "environmental",
                    "temperature",
                    "humidity",
                    "resolution",
                    "range",
                    "units",
                ]
            ):
                continue
            if not re.search(r"[a-zA-Z]", s):
                continue
            candidates.append((r, len(s), s))

    if not candidates:
        return None
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2]


def _sheet_has_nominal_measured(file_path, sheet_name, max_scan_rows=140):
    try:
        df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=object)
    except Exception:
        return False
    scan = min(max_scan_rows, df_raw.shape[0])
    for r in range(scan):
        row = [str(v).lower() if isinstance(v, str) else "" for v in df_raw.iloc[r]]
        txt = " ".join(row)
        if ("nominal value" in txt) and ("measured value" in txt):
            return True
    return False


def find_dimensional_result_sheet(xl, file_path):
    names = xl.sheet_names

    if "Results" in names and _sheet_has_nominal_measured(file_path, "Results"):
        return "Results"

    for candidate in ["inch", "mm", "inch (2)", "mm (2)"]:
        if candidate in names and _sheet_has_nominal_measured(file_path, candidate):
            return candidate

    for name in names:
        lower = name.lower()
        if any(k in lower for k in ["results", "mm", "inch"]):
            if _sheet_has_nominal_measured(file_path, name):
                return name

    return None


def parse_dimensional_table(file_path, sheet_name, gauge_id=None, unc_spec=None):
    try:
        df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=object)
    except Exception as e:
        print(f"   ⚠️ Could not read sheet '{sheet_name}': {e}")
        return pd.DataFrame()

    header_row = None
    for r in range(min(160, df_raw.shape[0])):
        row = [str(v).lower() if isinstance(v, str) else "" for v in df_raw.iloc[r]]
        if any("nominal value" in c for c in row) and any("measured value" in c for c in row):
            header_row = r
            break

    if header_row is None:
        print(f"   ⚠️ Could not find Nominal/Measured header in '{sheet_name}'")
        return pd.DataFrame()

    header = [str(v).lower() if isinstance(v, str) else "" for v in df_raw.iloc[header_row]]
    nominal_col = measured_col = deviation_col = None

    for idx, col in enumerate(header):
        if "nominal value" in col and nominal_col is None:
            nominal_col = idx
        if "measured value" in col and measured_col is None:
            measured_col = idx
        if "deviation" in col and deviation_col is None:
            deviation_col = idx

    if nominal_col is None or measured_col is None:
        print(f"   ⚠️ Nominal/Measured columns not found in '{sheet_name}'")
        return pd.DataFrame()

    unit = detect_unit(df_raw, header_row)
    feature_name = extract_feature_name(df_raw, header_row)

    records = []
    started = False

    for r in range(header_row + 1, df_raw.shape[0]):
        row = df_raw.iloc[r]
        nom_val = safe_numeric(row.iloc[nominal_col])
        meas_val = safe_numeric(row.iloc[measured_col])

        if nom_val is None and meas_val is None:
            if started:
                break
            continue

        started = True

        dev_val = safe_numeric(row.iloc[deviation_col]) if deviation_col is not None else None
        if dev_val is None and nom_val is not None and meas_val is not None:
            dev_val = meas_val - nom_val

        feature_index = str(r - header_row)
        meas_unc = compute_measurement_uncertainty(unc_spec, nom_val, unit)

        records.append(
            {
                "gauge_id": gauge_id,
                "feature_name": feature_name,
                "feature_index": feature_index,
                "nominal_value": nom_val,
                "measured_value": meas_val,
                "deviation": dev_val,
                "unit": unit,
                "measurement_uncertainty": meas_unc,
            }
        )

    if not records:
        print(f"   ⚠️ No numeric dimensional rows found in '{sheet_name}'")
        return pd.DataFrame()

    print(f"   ✅ Extracted {len(records)} dimensional rows from '{sheet_name}' (unit={unit}, feature={feature_name})")
    return pd.DataFrame(records)


def extract_dimensional_measurements(file_path, gauge_id, unc_spec):
    try:
        xl = pd.ExcelFile(file_path)
    except Exception as e:
        print(f"   ❌ Cannot open workbook for dimensional: {e}")
        return pd.DataFrame()

    sheet = find_dimensional_result_sheet(xl, file_path=file_path)
    if not sheet:
        print("   ⚠️ No usable dimensional results sheet found (no Nominal/Measured header detected)")
        return pd.DataFrame()

    print(f"   📊 Using dimensional sheet: '{sheet}'")
    return parse_dimensional_table(file_path, sheet, gauge_id=gauge_id, unc_spec=unc_spec)


# ============================================================
# TORQUE
# ============================================================
def detect_torque_direction(file_path):
    try:
        xl = pd.ExcelFile(file_path)
        text_all = ""
        for sheet in xl.sheet_names:
            if sheet.lower() in {"preset", "range", "beam"}:
                df = pd.read_excel(file_path, sheet_name=sheet, header=None, dtype=object)
                for v in df.values.ravel():
                    if isinstance(v, str):
                        text_all += " " + v.lower()
        cw = "clockwise direction" in text_all
        acw = "anti-clockwise direction" in text_all or "anticlockwise direction" in text_all
        if cw and acw:
            return "Clockwise & Anti-clockwise"
        if cw:
            return "Clockwise"
        if acw:
            return "Anti-clockwise"
        return None
    except Exception:
        return None


def parse_torque_beam_table(file_path, gauge_id=None, direction=None):
    try:
        xl = pd.ExcelFile(file_path)
    except Exception as e:
        print(f"   ❌ Cannot open workbook for torque: {e}")
        return pd.DataFrame()

    beam_sheet = None
    for name in xl.sheet_names:
        if "beam" in name.lower():
            beam_sheet = name
            break

    if beam_sheet is None:
        print("   ⚠️ No 'beam' sheet found for torque results.")
        return pd.DataFrame()

    try:
        df_raw = pd.read_excel(file_path, sheet_name=beam_sheet, header=None, dtype=object)
    except Exception as e:
        print(f"   ⚠️ Could not read beam sheet '{beam_sheet}': {e}")
        return pd.DataFrame()

    header_row = None
    for r in range(min(160, df_raw.shape[0])):
        row = [str(v).lower() if isinstance(v, str) else "" for v in df_raw.iloc[r]]
        joined = " ".join(row)
        if ("applied" in joined and "indicated" in joined and "error" in joined):
            header_row = r
            break

    if header_row is None:
        print("   ⚠️ Could not find torque header row on beam sheet.")
        return pd.DataFrame()

    header = [str(v).lower() if isinstance(v, str) else "" for v in df_raw.iloc[header_row]]

    applied_col = indicated_col = error_col = stddev_col = exp_unc_col = tol_spec_col = None
    for idx, col in enumerate(header):
        if "applied" in col and "torque" in col and applied_col is None:
            applied_col = idx
        if "indicated" in col and "torque" in col and indicated_col is None:
            indicated_col = idx
        if "error" in col and "indication" in col and error_col is None:
            error_col = idx
        if "standard deviation" in col and stddev_col is None:
            stddev_col = idx
        if "expanded uncertainty" in col and exp_unc_col is None:
            exp_unc_col = idx
        if "tolerance specification" in col and tol_spec_col is None:
            tol_spec_col = idx

    if applied_col is None or indicated_col is None:
        print("   ⚠️ Missing applied/indicated torque columns on beam sheet.")
        return pd.DataFrame()

    records = []
    started = False

    for r in range(header_row + 1, df_raw.shape[0]):
        row = df_raw.iloc[r]

        applied = safe_numeric(row.iloc[applied_col]) if applied_col is not None else None
        indicated = safe_numeric(row.iloc[indicated_col]) if indicated_col is not None else None
        error = safe_numeric(row.iloc[error_col]) if error_col is not None else None
        stddev = safe_numeric(row.iloc[stddev_col]) if stddev_col is not None else None
        exp_unc = safe_numeric(row.iloc[exp_unc_col]) if exp_unc_col is not None else None
        tol_spec = safe_numeric(row.iloc[tol_spec_col]) if tol_spec_col is not None else None

        is_trailing_blank = (
            started
            and (indicated is None)
            and (error is None)
            and (stddev is None)
            and (exp_unc is None)
            and (applied is None or applied == 0.0)
        )

        if not started:
            if applied is None and indicated is None and error is None and stddev is None and exp_unc is None and tol_spec is None:
                continue
            started = True

        if is_trailing_blank:
            break

        if error is None and applied is not None and indicated is not None:
            error = indicated - applied

        records.append(
            {
                "gauge_id": gauge_id,
                "direction": direction,
                "applied_torque_nm": applied,
                "indicated_torque_nm": indicated,
                "error_of_indication_nm": error,
                "standard_deviation_nm": stddev,
                "expanded_uncertainty_nm": exp_unc,
                "tolerance_specification_nm": tol_spec,
            }
        )

    if not records:
        print("   ⚠️ No torque rows extracted from beam sheet.")
        return pd.DataFrame()

    print(f"   ✅ Extracted {len(records)} torque rows from beam sheet '{beam_sheet}'")
    return pd.DataFrame(records)


# ============================================================
# SCHEMA (UNIVERSAL ONLY) + DEDUPE
# ============================================================
def ensure_schema(engine):
    """
    Universal calibration_metadata ONLY (no mass columns, no mass_metadata table).
    """
    with engine.begin() as conn:
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS calibration_metadata (
            id SERIAL PRIMARY KEY,
            file_name TEXT,
            certificate_number TEXT,
            calibration_date DATE,

            instrument_type TEXT,
            manufacturer TEXT,
            serial_number TEXT,
            previous_certificate_number TEXT,
            date_received DATE,
            calibration_procedure_number TEXT,
            method_used TEXT,
            comments TEXT,

            lab_type TEXT
        );
        """))

        # Add any missing universal columns
        for col, coltype in [
            ("file_name", "TEXT"),
            ("certificate_number", "TEXT"),
            ("calibration_date", "DATE"),
            ("instrument_type", "TEXT"),
            ("manufacturer", "TEXT"),
            ("serial_number", "TEXT"),
            ("previous_certificate_number", "TEXT"),
            ("date_received", "DATE"),
            ("calibration_procedure_number", "TEXT"),
            ("method_used", "TEXT"),
            ("comments", "TEXT"),
            ("lab_type", "TEXT"),
        ]:
            conn.execute(text(f"ALTER TABLE calibration_metadata ADD COLUMN IF NOT EXISTS {col} {coltype};"))

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS dimensional_results (
            id SERIAL PRIMARY KEY,
            metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
            certificate_number TEXT,
            gauge_id TEXT,
            feature_name TEXT,
            feature_index TEXT,
            nominal_value DOUBLE PRECISION,
            measured_value DOUBLE PRECISION,
            deviation DOUBLE PRECISION,
            unit TEXT,
            measurement_uncertainty DOUBLE PRECISION
        );
        """))

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS torque_results (
            id SERIAL PRIMARY KEY,
            metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
            certificate_number TEXT,
            gauge_id TEXT,
            direction TEXT,
            applied_torque_nm DOUBLE PRECISION,
            indicated_torque_nm DOUBLE PRECISION,
            error_of_indication_nm DOUBLE PRECISION,
            standard_deviation_nm DOUBLE PRECISION,
            expanded_uncertainty_nm DOUBLE PRECISION,
            tolerance_specification_nm DOUBLE PRECISION
        );
        """))

        # dedupe by file_name
        exists = conn.execute(text("""
            SELECT 1 FROM pg_constraint
            WHERE conname = 'uq_calibration_metadata_file_name'
        """)).scalar()
        if not exists:
            conn.execute(text("""
                ALTER TABLE calibration_metadata
                ADD CONSTRAINT uq_calibration_metadata_file_name UNIQUE (file_name);
            """))


# ============================================================
# IMPORT LOGIC
# ============================================================
META_COLS = [
    "file_name",
    "certificate_number",
    "calibration_date",
    "instrument_type",
    "manufacturer",
    "serial_number",
    "previous_certificate_number",
    "date_received",
    "calibration_procedure_number",
    "method_used",
    "comments",
    "lab_type",
]

def already_imported(engine, file_name: str) -> bool:
    with engine.connect() as conn:
        exists = conn.execute(
            text("SELECT 1 FROM calibration_metadata WHERE file_name=:fn LIMIT 1"),
            {"fn": file_name},
        ).scalar()
    return bool(exists)


def insert_data(engine, file_path):
    file_name = os.path.basename(file_path)

    if already_imported(engine, file_name):
        print(f"⏭️  Skipping (already imported): {file_name}")
        return True

    print(f"\n{'='*60}")
    print(f"📄 {file_name}")
    print("=" * 60)

    metadata = extract_metadata(file_path)
    gauge_id = extract_gauge_id(file_path)
    unc_spec = find_uncertainty_from_workbook(file_path)

    instrument_type = (metadata.get("instrument_type") or "").lower()
    lab_type = "Torque" if "torque" in instrument_type else "Dimensional"
    metadata["lab_type"] = lab_type

    # normalize text fields
    for k in [
        "manufacturer",
        "instrument_type",
        "serial_number",
        "previous_certificate_number",
        "calibration_procedure_number",
        "method_used",
        "comments",
        "certificate_number",
        "file_name",
    ]:
        if k in metadata:
            metadata[k] = clean_value(metadata.get(k))

    # --- WHITELIST metadata columns (prevents insert failures / accidental schema drift)
    md_row = {k: metadata.get(k) for k in META_COLS}

    # Insert metadata
    try:
        pd.DataFrame([md_row]).to_sql("calibration_metadata", engine, if_exists="append", index=False)

        with engine.connect() as conn:
            metadata_id = conn.execute(
                text("SELECT id FROM calibration_metadata WHERE file_name=:fn"),
                {"fn": file_name},
            ).scalar()

        print(f"✅ Metadata inserted (ID: {metadata_id}, lab_type={lab_type})")
    except Exception as e:
        msg = str(e).lower()
        if "uq_calibration_metadata_file_name" in msg or "duplicate key value" in msg:
            print(f"⏭️  Skipping (duplicate file_name): {file_name}")
            return True
        print(f"❌ Metadata insert error: {e}")
        return False

    cert_no = metadata.get("certificate_number")

    # Insert results
    try:
        if lab_type == "Dimensional":
            df = extract_dimensional_measurements(file_path, gauge_id=gauge_id, unc_spec=unc_spec)
            if df.empty:
                print("⚠️ No dimensional measurement data found (metadata still stored).")
                return True

            df["metadata_id"] = metadata_id
            df["certificate_number"] = cert_no
            df.to_sql("dimensional_results", engine, if_exists="append", index=False)
            print(f"✅ Inserted {len(df)} rows into 'dimensional_results'")

        else:  # Torque
            direction = detect_torque_direction(file_path)
            df = parse_torque_beam_table(file_path, gauge_id=gauge_id, direction=direction)
            if df.empty:
                print("⚠️ No torque rows found on beam sheet (metadata still stored).")
                return True

            df["metadata_id"] = metadata_id
            df["certificate_number"] = cert_no
            df.to_sql("torque_results", engine, if_exists="append", index=False)
            print(f"✅ Inserted {len(df)} rows into 'torque_results'")

        return True

    except Exception as e:
        print(f"❌ Insert error (results tables): {e}")
        return False


# ============================================================
# MAIN
# ============================================================
def run_dimensional_torque_import(folder_path=".", ensure_db_schema=True):
    print("🚀 NML Dimensional + Torque Import Tool (Universal metadata + dedupe)")
    print("=" * 70)

    try:
        engine = create_engine(DB_URL)
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        print("✅ Database connected\n")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return

    if ensure_db_schema:
        ensure_schema(engine)

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(".xlsm") and not f.startswith("~$")]
    if not files:
        print(f"⚠️ No .xlsm files found in '{folder_path}'")
        return

    print(f"Found {len(files)} file(s)\n")

    success = 0
    for f in sorted(files):
        if insert_data(engine, os.path.join(folder_path, f)):
            success += 1

    print(f"\n{'='*70}")
    print("📊 SUMMARY")
    print("=" * 70)
    print(f"Seen in folder: {len(files)}")
    print(f"Handled successfully: {success}")
    print(f"Errors: {len(files) - success}")

    try:
        with engine.connect() as conn:
            for table in ["calibration_metadata", "dimensional_results", "torque_results"]:
                cnt = conn.execute(text(f"SELECT COUNT(*) FROM public.{table}")).scalar()
                print(f"   public.{table}: {cnt} records")
    except Exception as e:
        print(f"\n⚠️ Count error: {e}")

    print("\n✅ Import complete!")


if __name__ == "__main__":
    run_dimensional_torque_import(folder_path=".", ensure_db_schema=True)

🚀 NML Dimensional + Torque Import Tool (Universal metadata + dedupe)
✅ Database connected

Found 5 file(s)

⏭️  Skipping (already imported): 24-8443 Results.xlsm
⏭️  Skipping (already imported): 24-8444 Results.xlsm
⏭️  Skipping (already imported): 24-8782.xlsm
⏭️  Skipping (already imported): 24-8809.xlsm
⏭️  Skipping (already imported): 24-8859 Results.xlsm

📊 SUMMARY
Seen in folder: 5
Handled successfully: 5
Errors: 0
   public.calibration_metadata: 87 records
   public.dimensional_results: 32 records
   public.torque_results: 11 records

✅ Import complete!
